# Proof Pilot — env smoke test (no model)

Validates the **`proof-pilot-env`** dataset is usable on this GPU **without** the 32B model.
It extracts the archive, relocates the venv (stdlib comes from the bundled standalone CPython),
checks the GPU + CUDA, imports sglang/torch/flashinfer/triton, runs a real CUDA kernel, imports
the humming W4A8 glue, and verifies the sglang patches are baked into the venv.

**Usage:** add the `proof-pilot-env` dataset, set `DS_ENV` below if your slug differs, then *Run All*.
A green **ALL PASS** at the bottom means the env is good to pair with the model dataset.

In [ ]:
# --- 1. locate + extract the env archive into writable scratch (Kaggle input is read-only) ---
import os, subprocess, glob, re, json, textwrap, shutil

WORK = os.environ.get("WORK", "/tmp/pp")
VENV, PYBASE = f"{WORK}/venv", f"{WORK}/pybase"
os.makedirs(WORK, exist_ok=True)

def find_archive():
    # explicit DS_ENV first, then common Kaggle mount paths, then a recursive search.
    # the archive ships under an opaque name (proof-pilot-env.bin) so Kaggle stores it as a blob
    # instead of auto-extracting it; we identify it by content, so match the name loosely.
    cands = ([os.environ["DS_ENV"]] if os.environ.get("DS_ENV") else []) + [
        "/kaggle/input/proof-pilot-env",
        "/kaggle/input/datasets/threerabbits/proof-pilot-env"]
    pats = ("proof-pilot-env.bin", "proof-pilot-env.tar.*", "*.bin", "*.tar.gz", "*.tar.zst")
    for d in cands:
        for p in pats:
            g = sorted(glob.glob(f"{d}/{p}"))
            if g: return d, g[0]
    g = sorted(glob.glob("/kaggle/input/**/proof-pilot-env.*", recursive=True))
    g = [x for x in g if x.endswith((".bin", ".tar.gz", ".tar.zst"))]
    return (os.path.dirname(g[0]), g[0]) if g else (None, None)

if not os.path.exists(f"{VENV}/bin/python"):
    DS_ENV, arc = find_archive()
    assert arc, "env archive not found — add the proof-pilot-env dataset to this notebook"
    magic = open(arc, "rb").read(4)
    print("DS_ENV =", DS_ENV, "| archive =", os.path.basename(arc), "| magic =", magic.hex(), "| WORK =", WORK)
    if magic[:2] == b"\x1f\x8b":             # gzip — universal (Kaggle has no zstd)
        subprocess.run(["tar", "-xzf", arc, "-C", WORK, "--strip-components=1"], check=True)
    elif magic == b"\x28\xb5\x2f\xfd":       # zstd — only where a zstd binary exists
        assert shutil.which("zstd"), "zstd archive but no zstd binary here — re-pack as gzip"
        subprocess.run(["tar", "-x", "-I", "zstd -d --long=31", "-f", arc,
                        "-C", WORK, "--strip-components=1"], check=True)
    else:
        raise SystemExit(f"unknown archive magic {magic!r} for {arc}")
else:
    print("venv already staged at", VENV)
print("top-level:", sorted(os.listdir(WORK)))
assert os.path.exists(f"{VENV}/bin/python"), "venv/bin/python missing after extract"

In [ ]:
# --- 2. relocate the venv (stdlib in pybase) + restore warm JIT caches ---
cfg = f"{VENV}/pyvenv.cfg"
txt = re.sub(r"(?m)^home = .*$", f"home = {PYBASE}/bin", open(cfg).read())
open(cfg, "w").write(txt)
print("pyvenv:", next(l for l in txt.splitlines() if l.startswith("home")))

HOME = os.path.expanduser("~")
for src, dst in [(f"{WORK}/flashinfer_cache", f"{HOME}/.cache/flashinfer"),
                 (f"{WORK}/humming_cache",    f"{HOME}/.humming/cache")]:
    if os.path.isdir(src):
        os.makedirs(dst, exist_ok=True)
        subprocess.run(["cp", "-rn", f"{src}/.", f"{dst}/"])
        print("cache ->", dst, "(", len(os.listdir(dst)), "entries )")

In [ ]:
# --- 3. run the import / GPU / CUDA-exec / humming battery INSIDE the relocated venv ---
CHECK = textwrap.dedent(r"""
    import sys, os, json, glob, ctypes
    R = {}
    def chk(k, cond, info=""): R[k] = {"pass": bool(cond), "info": str(info)}

    # stdlib resolves from the bundled pybase (not the build host's /.uv path)
    chk("stdlib_from_pybase",
        os.path.realpath(sys.base_prefix) == os.path.realpath(os.environ["PYBASE"]),
        sys.base_prefix)

    import torch;      chk("import torch", True, torch.__version__)
    import sglang;     chk("import sglang", True, sglang.__version__)
    import flashinfer; chk("import flashinfer", True, getattr(flashinfer, "__version__", "?"))
    import triton;     chk("import triton", True, triton.__version__)

    chk("cuda_available", torch.cuda.is_available())
    if torch.cuda.is_available():
        cc = torch.cuda.get_device_capability(0); name = torch.cuda.get_device_name(0)
        chk("gpu_blackwell_sm120", cc == (12, 0), f"{name} cc={cc[0]}.{cc[1]}")
        x = torch.randn(2048, 2048, device="cuda", dtype=torch.bfloat16)
        y = x @ x; torch.cuda.synchronize()       # real kernel exec catches arch 'no kernel image'
        chk("cuda_kernel_exec", bool(torch.isfinite(y).all()), f"bf16 matmul {tuple(y.shape)} ok")

    # humming W4A8 glue: preload nvrtc GLOBAL (like serve_final's LD_PRELOAD), then import
    try:
        nv = glob.glob(os.path.join(sys.prefix, "lib/python*/site-packages/nvidia/cu13/lib/libnvrtc.so.13"))
        if nv: ctypes.CDLL(nv[0], mode=ctypes.RTLD_GLOBAL)
        sys.path.insert(0, os.environ["W4A8_HELPER_DIR"])
        import humming_w4a8 as h
        h._lazy_import()                          # imports the humming package via HUMMING_PATH
        chk("humming_glue_import", True, "humming + w4a8 glue OK")
    except Exception as e:
        chk("humming_glue_import", False, repr(e))
    ncache = len(glob.glob(os.path.expanduser("~/.humming/cache/*")))
    chk("humming_cubin_cache", ncache > 0, f"{ncache} cubins")

    print("RESULT_JSON=" + json.dumps(R))
""")
env = dict(os.environ, PYBASE=PYBASE, HUMMING_PATH=WORK,
           W4A8_HELPER_DIR=f"{WORK}/proof-pilot/deploy/w4a8")
p = subprocess.run([f"{VENV}/bin/python", "-c", CHECK], env=env, capture_output=True, text=True)
print(p.stdout)
if p.returncode != 0:
    print("--- STDERR (tail) ---\n", p.stderr[-3000:])
res = {}
for line in p.stdout.splitlines():
    if line.startswith("RESULT_JSON="):
        res = json.loads(line[len("RESULT_JSON="):])

In [ ]:
# --- 4. verify the sglang patches are baked into the venv (verify-only, no writes) ---
REPO = f"{WORK}/proof-pilot"
p = subprocess.run(["bash", f"{REPO}/kaggle/serve/apply_all_patches.sh", VENV, "--verify-only"],
                   capture_output=True, text=True)
print(p.stdout[-2500:])
if p.returncode: print("--- stderr (tail) ---\n", p.stderr[-1500:])
# direct cross-check on the actual venv files (robust signal independent of the script's exit code)
import glob as _g
wna = _g.glob(f"{VENV}/lib/python*/site-packages/sglang/srt/layers/quantization/compressed_tensors/schemes/compressed_tensors_wNa16.py")
humming_baked = bool(wna) and ("humming" in open(wna[0]).read().lower())
patch_ok = humming_baked and ("OK" in p.stdout or "applied" in p.stdout)
print("humming patch baked in venv:", humming_baked)

In [ ]:
# --- 5. summary ---
allres = dict(res)
allres["patches_baked"] = {"pass": bool(patch_ok), "info": "apply_all_patches --verify-only + venv grep"}
print("=== ENV SMOKE TEST ===")
fails = []
for k, v in allres.items():
    print(("  PASS" if v["pass"] else "  FAIL"), k, "-", v["info"])
    if not v["pass"]: fails.append(k)
print()
if not fails:
    print("ALL PASS - env usable on this GPU; ready to pair with the model dataset")
else:
    print(f"{len(fails)} FAILED: {fails}")
assert not fails, f"env smoke test failed: {fails}"